# 序列模型高级应用

本notebook介绍RNN的高级应用:
- **双向RNN**: 同时利用过去和未来
- **深层RNN**: 堆叠多层提升表达能力
- **Seq2Seq**: 序列到序列学习
- **机器翻译**: 实际应用案例

这些技术是NLP任务的基础!

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
import collections

---

## 第一部分: 双向RNN

### 1.1 单向RNN的局限

**问题**: 只能看到过去

```
"我爱____" → 预测下一个字
只能基于"我爱"来预测
```

**但有时未来也重要**:
```
"我___看电影" → 填空
需要同时看"我"和"看电影"
答案可能是"想"或"常"
```

### 1.2 双向RNN原理

**前向RNN**:
$$\overrightarrow{\mathbf{H}}_t = \phi(\mathbf{X}_t \mathbf{W}_{xh}^{(f)} + \overrightarrow{\mathbf{H}}_{t-1} \mathbf{W}_{hh}^{(f)} + \mathbf{b}_h^{(f)})$$

**后向RNN**:
$$\overleftarrow{\mathbf{H}}_t = \phi(\mathbf{X}_t \mathbf{W}_{xh}^{(b)} + \overleftarrow{\mathbf{H}}_{t+1} \mathbf{W}_{hh}^{(b)} + \mathbf{b}_h^{(b)})$$

**拼接输出**:
$$\mathbf{H}_t = [\overrightarrow{\mathbf{H}}_t; \overleftarrow{\mathbf{H}}_t]$$
$$\mathbf{O}_t = \mathbf{H}_t \mathbf{W}_{hq} + \mathbf{b}_q$$

### 1.3 双向RNN结构

```
时间步:    1       2       3       4
           ↓       ↓       ↓       ↓
输入:     x_1     x_2     x_3     x_4
           ↓       ↓       ↓       ↓
前向:   →H_1  →  H_2  →  H_3  →  H_4 →
后向:  ← H_1  ← H_2  ←  H_3  ←  H_4 ←
           ↓       ↓       ↓       ↓
拼接:    [→;←]  [→;←]  [→;←]  [→;←]
           ↓       ↓       ↓       ↓
输出:     o_1     o_2     o_3     o_4
```

**注意**: 需要完整序列,不适合在线预测!

### 1.4 实现双向RNN

In [ ]:
class BiRNNModel(nn.Module):
    """双向RNN模型"""
    def __init__(self, vocab_size, embed_size, num_hiddens, num_layers):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        
        # bidirectional=True
        self.rnn = nn.LSTM(embed_size, num_hiddens, num_layers,
                           bidirectional=True, batch_first=True)
        
        # 输出层: 2*num_hiddens (前向+后向)
        self.fc = nn.Linear(2 * num_hiddens, vocab_size)
    
    def forward(self, x):
        # x: (batch, seq_len)
        x = self.embedding(x)  # (batch, seq_len, embed)
        output, _ = self.rnn(x)  # (batch, seq_len, 2*hidden)
        output = self.fc(output)  # (batch, seq_len, vocab)
        return output


# 测试
model = BiRNNModel(vocab_size=1000, embed_size=128, 
                   num_hiddens=256, num_layers=2)
print(model)

# 前向传播
x = torch.randint(0, 1000, (2, 5))  # (batch=2, seq_len=5)
output = model(x)
print(f"\n输入: {x.shape}")
print(f"输出: {output.shape}")  # (2, 5, 1000)

# 参数量对比
uni_model = nn.LSTM(128, 256, 2, bidirectional=False)
bi_model = nn.LSTM(128, 256, 2, bidirectional=True)
print(f"\n单向参数: {sum(p.numel() for p in uni_model.parameters()):,}")
print(f"双向参数: {sum(p.numel() for p in bi_model.parameters()):,}")
print("双向RNN参数约为单向的2倍")

### 1.5 双向RNN应用

**适合**:
- ✅ 填空任务
- ✅ 情感分析(完整句子)
- ✅ 命名实体识别
- ✅ 词性标注

**不适合**:
- ❌ 语言模型(只能看过去)
- ❌ 在线预测
- ❌ 实时应用

---

## 第二部分: 深层RNN

### 2.1 为什么要堆叠?

**单层RNN**:
- 表达能力有限
- 学习简单模式

**多层RNN**:
- 更强的表达能力
- 层级表示
- 底层→语法,高层→语义

### 2.2 深层RNN结构

```
输入:   x_1    x_2    x_3
         ↓      ↓      ↓
Layer1: H¹_1 → H¹_2 → H¹_3
         ↓      ↓      ↓
Layer2: H²_1 → H²_2 → H²_3
         ↓      ↓      ↓
Layer3: H³_1 → H³_2 → H³_3
         ↓      ↓      ↓
输出:   o_1    o_2    o_3
```

**公式**:
$$\mathbf{H}_t^{(l)} = \phi(\mathbf{H}_t^{(l-1)} \mathbf{W}_{xh}^{(l)} + \mathbf{H}_{t-1}^{(l)} \mathbf{W}_{hh}^{(l)} + \mathbf{b}_h^{(l)})$$

其中 $\mathbf{H}_t^{(0)} = \mathbf{X}_t$

### 2.3 实现深层RNN

In [ ]:
class DeepRNNModel(nn.Module):
    """深层RNN模型"""
    def __init__(self, vocab_size, embed_size, num_hiddens, num_layers, dropout=0.5):
        super().__init__()
        self.num_layers = num_layers
        
        self.embedding = nn.Embedding(vocab_size, embed_size)
        
        # num_layers层LSTM
        self.rnn = nn.LSTM(embed_size, num_hiddens, num_layers,
                           dropout=dropout,  # 层间dropout
                           batch_first=True)
        
        self.fc = nn.Linear(num_hiddens, vocab_size)
    
    def forward(self, x):
        x = self.embedding(x)
        output, _ = self.rnn(x)
        output = self.fc(output)
        return output


# 对比不同层数
for num_layers in [1, 2, 3, 4]:
    model = DeepRNNModel(vocab_size=1000, embed_size=128,
                         num_hiddens=256, num_layers=num_layers)
    params = sum(p.numel() for p in model.parameters())
    print(f"{num_layers}层RNN参数量: {params:,}")

print("\n注意: 参数量线性增长")

### 2.4 层数选择

| 层数 | 适用场景 | 注意 |
|------|----------|------|
| 1层 | 简单任务 | 够用 |
| 2层 | 大多数任务 | **推荐** |
| 3-4层 | 复杂任务 | 需要更多数据 |
| 5+层 | 极少使用 | 容易过拟合 |

**经验法则**: 2层通常够用!

---

## 第三部分: Seq2Seq (序列到序列)

### 3.1 Seq2Seq的动机

**问题**: 输入输出长度不同

**例子**:
- 机器翻译: "我爱你" (3) → "I love you" (3) ✓
- 但: "今天天气真好" (6) → "Nice weather today" (3) ?

**Seq2Seq方案**: 编码器-解码器架构

### 3.2 编码器-解码器架构

```
编码器 (Encoder):     解码器 (Decoder):
输入序列 → 上下文向量 → 输出序列

"我爱你"   →   [c]   →  "I love you"
(源语言)      (固定向量)   (目标语言)
```

**核心思想**: 压缩 → 解压

### 3.3 编码器

**任务**: 序列 → 固定长度向量

$$\mathbf{h}_t = f(\mathbf{x}_t, \mathbf{h}_{t-1})$$
$$\mathbf{c} = q(\mathbf{h}_1, ..., \mathbf{h}_T)$$

通常: $\mathbf{c} = \mathbf{h}_T$ (最后隐状态)

### 3.4 解码器

**任务**: 固定向量 → 序列

$$\mathbf{s}_t = g(\mathbf{y}_{t-1}, \mathbf{c}, \mathbf{s}_{t-1})$$
$$P(y_t \mid y_1, ..., y_{t-1}, \mathbf{c}) = \text{softmax}(\mathbf{s}_t \mathbf{W})$$

**特殊token**:
- `<bos>`: 开始
- `<eos>`: 结束
- `<pad>`: 填充

### 3.5 实现Seq2Seq

In [ ]:
class Encoder(nn.Module):
    """Seq2Seq编码器"""
    def __init__(self, vocab_size, embed_size, num_hiddens, num_layers, dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.rnn = nn.LSTM(embed_size, num_hiddens, num_layers,
                           dropout=dropout, batch_first=True)
    
    def forward(self, x):
        # x: (batch, seq_len)
        x = self.embedding(x)  # (batch, seq_len, embed)
        output, state = self.rnn(x)
        # output: (batch, seq_len, hidden)
        # state: (h, c), h/c: (num_layers, batch, hidden)
        return output, state


class Decoder(nn.Module):
    """Seq2Seq解码器"""
    def __init__(self, vocab_size, embed_size, num_hiddens, num_layers, dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        # 输入: embedding + 上下文(encoder最后隐状态)
        self.rnn = nn.LSTM(embed_size + num_hiddens, num_hiddens, 
                           num_layers, dropout=dropout, batch_first=True)
        self.fc = nn.Linear(num_hiddens, vocab_size)
    
    def forward(self, x, state):
        # x: (batch, seq_len)
        x = self.embedding(x)  # (batch, seq_len, embed)
        
        # 上下文向量(编码器最后隐状态)
        context = state[0][-1].unsqueeze(1)  # (batch, 1, hidden)
        # 重复到每个时间步
        context = context.repeat(1, x.size(1), 1)  # (batch, seq_len, hidden)
        
        # 拼接输入和上下文
        x = torch.cat([x, context], dim=2)  # (batch, seq_len, embed+hidden)
        
        output, state = self.rnn(x, state)
        output = self.fc(output)  # (batch, seq_len, vocab)
        return output, state


class Seq2Seq(nn.Module):
    """完整的Seq2Seq模型"""
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
    
    def forward(self, src, tgt):
        # 编码
        _, state = self.encoder(src)
        # 解码
        output, _ = self.decoder(tgt, state)
        return output


# 创建模型
src_vocab = 1000  # 源语言词表
tgt_vocab = 800   # 目标语言词表
embed_size = 128
num_hiddens = 256
num_layers = 2

encoder = Encoder(src_vocab, embed_size, num_hiddens, num_layers)
decoder = Decoder(tgt_vocab, embed_size, num_hiddens, num_layers)
model = Seq2Seq(encoder, decoder)

print(model)
print(f"\n总参数量: {sum(p.numel() for p in model.parameters()):,}")

# 测试
src = torch.randint(0, src_vocab, (2, 5))  # batch=2, src_len=5
tgt = torch.randint(0, tgt_vocab, (2, 7))  # batch=2, tgt_len=7
output = model(src, tgt)
print(f"\n源序列: {src.shape}")
print(f"目标序列: {tgt.shape}")
print(f"输出: {output.shape}")  # (2, 7, 800)

### 3.6 训练Seq2Seq

**Teacher Forcing**:
- 训练时: 用真实目标序列作为解码器输入
- 推理时: 用模型自己的预测作为输入

```python
# 训练
for epoch in range(num_epochs):
    for src, tgt in dataloader:
        # tgt_input: <bos> + tgt[:-1]
        # tgt_output: tgt + <eos>
        output = model(src, tgt_input)
        loss = criterion(output, tgt_output)
        loss.backward()
        optimizer.step()
```

### 3.7 Seq2Seq的问题

**瓶颈问题**:
- 所有信息压缩到固定向量 $\mathbf{c}$
- 长序列信息丢失
- 性能下降

**解决方案**: **注意力机制** (下一章!)

---

## 第四部分: 机器翻译示例

### 4.1 数据预处理

In [ ]:
# 简单的中英翻译数据
pairs = [
    ("我 爱 你", "I love you"),
    ("今天 天气 很 好", "The weather is nice today"),
    ("我 喜欢 学习", "I like studying"),
    ("这 是 一 本 书", "This is a book"),
    ("你 好 吗", "How are you"),
]

# 构建词表
def build_vocab(sentences):
    vocab = {'<pad>': 0, '<bos>': 1, '<eos>': 2, '<unk>': 3}
    for sent in sentences:
        for word in sent.split():
            if word not in vocab:
                vocab[word] = len(vocab)
    return vocab

src_sents = [p[0] for p in pairs]
tgt_sents = [p[1] for p in pairs]

src_vocab = build_vocab(src_sents)
tgt_vocab = build_vocab(tgt_sents)

print("源语言词表:", src_vocab)
print("\n目标语言词表:", tgt_vocab)

# 数值化
def encode(sentence, vocab):
    return [vocab.get(w, vocab['<unk>']) for w in sentence.split()]

# 示例
src_encoded = encode("我 爱 你", src_vocab)
tgt_encoded = encode("I love you", tgt_vocab)
print(f"\n源序列编码: {src_encoded}")
print(f"目标序列编码: {tgt_encoded}")

### 4.2 推理(贪心解码)

In [ ]:
def greedy_decode(model, src, src_vocab, tgt_vocab, max_len=20):
    """贪心解码"""
    model.eval()
    with torch.no_grad():
        # 编码
        _, state = model.encoder(src)
        
        # 解码: 从<bos>开始
        tgt = torch.tensor([[tgt_vocab['<bos>']]], device=src.device)
        result = []
        
        for _ in range(max_len):
            output, state = model.decoder(tgt, state)
            # 贪心选择
            pred = output.argmax(dim=2)[:, -1:]
            result.append(pred.item())
            
            # 遇到<eos>停止
            if pred.item() == tgt_vocab['<eos>']:
                break
            
            # 下一步输入
            tgt = torch.cat([tgt, pred], dim=1)
        
        return result

# 创建反向词表
def get_inv_vocab(vocab):
    return {v: k for k, v in vocab.items()}

tgt_inv_vocab = get_inv_vocab(tgt_vocab)

# 示例(未训练)
src_seq = torch.tensor([[src_vocab['我'], src_vocab['爱'], src_vocab['你']]])
pred_ids = greedy_decode(model, src_seq, src_vocab, tgt_vocab)
pred_words = [tgt_inv_vocab.get(i, '<unk>') for i in pred_ids]
print(f"输入: 我 爱 你")
print(f"预测(未训练): {' '.join(pred_words)}")

---

## 小结

### 核心技术

1. **双向RNN**: 同时利用过去和未来
   - 前向 + 后向
   - 拼接隐状态
   - 适合填空/分类任务

2. **深层RNN**: 堆叠多层
   - 层级表示
   - 更强表达能力
   - 2层通常够用

3. **Seq2Seq**: 编码器-解码器
   - 处理变长序列
   - 机器翻译
   - 需要注意力机制改进

### 应用场景

| 任务 | 架构 | 关键点 |
|------|------|--------|
| 情感分析 | 双向RNN | 需要完整句子 |
| 命名实体识别 | 双向RNN + CRF | 序列标注 |
| 机器翻译 | Seq2Seq + Attention | 变长输入输出 |
| 文本摘要 | Seq2Seq | 长→短 |
| 对话系统 | Seq2Seq | 上下文建模 |

### Seq2Seq的演进

```
基础Seq2Seq
  ↓
Seq2Seq + Attention (解决瓶颈)
  ↓
Transformer (完全基于注意力)
  ↓
BERT/GPT (预训练模型)
```

### 实践建议

1. **双向RNN**: 离线任务首选
2. **层数**: 2层是甜点
3. **Seq2Seq**: 必须加注意力机制
4. **解码**: Beam Search优于贪心
5. **数据**: 对齐很重要

## 练习

1. 实现双向GRU用于情感分析
2. 对比1层/2层/3层RNN的性能
3. 在中英翻译数据上训练Seq2Seq
4. 实现Beam Search解码
5. 可视化编码器的隐状态